In [18]:
from Bio import SeqIO
import numpy as np
import pandas as pd
from collections import Counter

In [19]:
proteins = []
file_path = 'deeploc_data.fasta'
file_type = 'fasta'
for protein in SeqIO.parse(file_path, file_type):
    protein_seq = str(protein.seq)

    # header
    header = protein.description

    proteins.append({
        "protein_id": protein.id,
        "sequence": protein_seq,
        "header": header
    })

df = pd.DataFrame(proteins)
print(df.head())

  protein_id                                           sequence  \
0     Q9H400  MGLPVSWAPPALWVLGCCALLLSLWALCTACRRPEDAVAPRKRARR...   
1     Q5I0E9  MEVLEEPAPGPGGADAAERRGLRRLLLSGFQEELRALLVLAGPAFL...   
2     P63033  MMKTLSSGNCTLNVPAKNSYRMVVLGASRVGKSSIVSRFLNGRFED...   
3     Q9NR71  MAKRTFSNLETFLIFLLVMMSAITVALLSLLFITSGTIENHKDLGG...   
4     Q86XT9  MGNCQAGHNLHLCLAHHPPLVCATLILLLLGLSGLGLGSFLLTHRT...   

                        header  
0  Q9H400 Cell.membrane-M test  
1       Q5I0E9 Cell.membrane-M  
2       P63033 Cell.membrane-M  
3       Q9NR71 Cell.membrane-M  
4       Q86XT9 Cell.membrane-M  


In [20]:
def parse_header(header):
    '''
    Extract the subcellular location (output label), membrane_status, and train/test label from the header
    Args:
        header(str): header for row
    Returns:
        Tuple of subcellular loc, membrane status, and train test label

    '''
    header_parts = header.split()

    subcellular_location = header_parts[1][:-2]
    membrane_status = header_parts[1][-1]

    if len(header_parts) > 2:
        train_test = header_parts[2]
    else:
        train_test = 'train'
    
    return (subcellular_location, membrane_status, train_test)

new_cols = df['header'].apply(parse_header).apply(pd.Series)
df[['subcellular_location', 'membrane_status', 'train_test']] = new_cols

# drop headers column: info has been parsed into new cols
df.drop(columns=['header'], inplace=True)

print(df.head())

  protein_id                                           sequence  \
0     Q9H400  MGLPVSWAPPALWVLGCCALLLSLWALCTACRRPEDAVAPRKRARR...   
1     Q5I0E9  MEVLEEPAPGPGGADAAERRGLRRLLLSGFQEELRALLVLAGPAFL...   
2     P63033  MMKTLSSGNCTLNVPAKNSYRMVVLGASRVGKSSIVSRFLNGRFED...   
3     Q9NR71  MAKRTFSNLETFLIFLLVMMSAITVALLSLLFITSGTIENHKDLGG...   
4     Q86XT9  MGNCQAGHNLHLCLAHHPPLVCATLILLLLGLSGLGLGSFLLTHRT...   

  subcellular_location membrane_status train_test  
0        Cell.membrane               M       test  
1        Cell.membrane               M      train  
2        Cell.membrane               M      train  
3        Cell.membrane               M      train  
4        Cell.membrane               M      train  


In [21]:
df['train_test'].value_counts()

train_test
train    11231
test      2773
Name: count, dtype: int64

In [22]:
# check missing values
print(df.isnull().sum())
# check duplicates
print("Duplicate IDs:", df['protein_id'].duplicated().sum())
print("Duplicate sequences:", df['sequence'].duplicated().sum())

protein_id              0
sequence                0
subcellular_location    0
membrane_status         0
train_test              0
dtype: int64
Duplicate IDs: 0
Duplicate sequences: 55


In [23]:
# check unusual amino acids
usual_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")
unusual_amino_acids = set("XBZUOJ")

# check if a sequence contains unusual amino acids
def count_unusual(seq):
    count = 0
    for aa in seq:
        if aa in unusual_amino_acids:
            count += 1
    return count

df['num_unusual_aa'] = df['sequence'].apply(count_unusual)
print((df['num_unusual_aa'] > 0).sum())

34


In [24]:
from collections import Counter

unusual_rows = df[df['num_unusual_aa'] > 0]

counts = Counter()

for seq in unusual_rows['sequence']:
    for aa in seq:
        if aa in unusual_amino_acids:
            counts[aa] += 1

print(counts)

Counter({'X': 76, 'U': 30, 'Z': 4, 'B': 2})


In [25]:
# filter out rows with unusual amino acids
df = df[df['num_unusual_aa'] == 0].copy()

In [26]:
# check variation in sequence lengths
df['seq_length'] = df['sequence'].str.len()
print(df['seq_length'].describe())

count    13970.000000
mean       522.662133
std        448.242730
min         40.000000
25%        250.000000
50%        422.000000
75%        649.000000
max      13100.000000
Name: seq_length, dtype: float64


In [27]:
# inspect duplicates
duplicates = df[df.duplicated('sequence', keep=False)]
duplicates = duplicates.sort_values('sequence')
print(f"duplicate labels: {duplicates['train_test'].value_counts()}")

print(duplicates[
    ['protein_id', 'subcellular_location',
     'membrane_status', 'train_test']
])

cols = ['subcellular_location', 'membrane_status', 'train_test']

# check which duplicates have mismatch data: potential problem
conflicts = df.groupby('sequence')[cols].nunique()
conflicting_sequences = conflicts[(conflicts > 1).any(axis=1)]

# retreive df rows of duplicates
conflicting_rows = df[df['sequence'].isin(conflicting_sequences.index)]

print(conflicting_rows[['protein_id', 'subcellular_location', 'membrane_status', 'train_test']])

duplicate labels: train_test
train    95
test     10
Name: count, dtype: int64
      protein_id subcellular_location membrane_status train_test
11674     P68164              Plastid               U      train
11675     P68163              Plastid               U      train
13583     P0C892        Extracellular               S      train
13018     Q86QV0        Extracellular               S      train
13898     P83288        Extracellular               S      train
...          ...                  ...             ...        ...
1628      P0CX27            Cytoplasm               S      train
13421     Q25619        Extracellular               S      train
13962     Q8WT57        Extracellular               S      train
13008     P85174        Extracellular               S      train
13654     P85436        Extracellular               S      train

[105 rows x 4 columns]
Empty DataFrame
Columns: [protein_id, subcellular_location, membrane_status, train_test]
Index: []


In [28]:
# separate train and test dataframes
train_df = df[df['train_test'] == 'train'].copy()
test_df = df[df['train_test'] == 'test'].copy()

# remove duplicates from training data
train_df = train_df.drop_duplicates(subset='sequence')

# train and test dataset sizes
print("Training set size:", len(train_df))
print("Test set size:", len(test_df))
train_percent = 100 * len(train_df) / len(df)
test_percent = 100 * len(test_df) / len(df)
print("Train/Test Split:", train_percent, ':', test_percent)

Training set size: 11151
Test set size: 2769
Train/Test Split: 79.82104509663564 : 19.821045096635647


In [29]:
train_df.head()

,protein_id,sequence,subcellular_location,membrane_status,train_test,num_unusual_aa,seq_length
1,Q5I0E9,MEVLEEPAPGPGGADAAERRGLRRLLLSGFQEELRALLVLAGPAFL...,Cell.membrane,M,train,0,566
2,P63033,MMKTLSSGNCTLNVPAKNSYRMVVLGASRVGKSSIVSRFLNGRFED...,Cell.membrane,M,train,0,266
3,Q9NR71,MAKRTFSNLETFLIFLLVMMSAITVALLSLLFITSGTIENHKDLGG...,Cell.membrane,M,train,0,780
4,Q86XT9,MGNCQAGHNLHLCLAHHPPLVCATLILLLLGLSGLGLGSFLLTHRT...,Cell.membrane,M,train,0,240
5,A2CI98,MDPSKQGTLNRVENSVYRTAFKLRSVQTLCQLDLMDSFLIQQVLWR...,Cell.membrane,M,train,0,653


In [30]:
test_df.head()

,protein_id,sequence,subcellular_location,membrane_status,train_test,num_unusual_aa,seq_length
0,Q9H400,MGLPVSWAPPALWVLGCCALLLSLWALCTACRRPEDAVAPRKRARR...,Cell.membrane,M,test,0,295
9,P83456,AGFPEQEPEPKFWNDWAQKTLDKALSLQTLNKNKAQNLILFLGDGM...,Cell.membrane,M,test,0,477
15,Q9GL77,MEDEAVLDRGASFLKHVCDEEEVEGHHTIYIGVHVPKSYRRRRRHK...,Cell.membrane,M,test,0,1079
20,Q03445,MHSRLKFLAYLHFICASSIFWPEFSSAQQQQQTVSLTEKIPLGAIF...,Cell.membrane,M,test,0,991
23,P35525,MAAATAAAATVAGEGMEPRALQYEQTLMYGRYTQELGAFAKEEAAR...,Cell.membrane,M,test,0,907


In [31]:
# check for data leakage

train_sequences = set(train_df["sequence"])
test_sequences = set(test_df["sequence"])

overlaping_sequences = train_sequences & test_sequences

print("Sequence overlap in train/test set: ", len(overlaping_sequences))

Sequence overlap in train/test set:  0


In [32]:
# inspect output class balance

print(train_df["subcellular_location"].value_counts())
print(train_df["subcellular_location"].value_counts(normalize=True))

subcellular_location
Nucleus                  3225
Cytoplasm                2013
Extracellular            1553
Mitochondrion            1199
Cell.membrane            1065
Endoplasmic.reticulum     686
Plastid                   601
Golgi.apparatus           285
Lysosome/Vacuole          255
Cytoplasm-Nucleus         145
Peroxisome                124
Name: count, dtype: int64
subcellular_location
Nucleus                  0.289212
Cytoplasm                0.180522
Extracellular            0.139270
Mitochondrion            0.107524
Cell.membrane            0.095507
Endoplasmic.reticulum    0.061519
Plastid                  0.053897
Golgi.apparatus          0.025558
Lysosome/Vacuole         0.022868
Cytoplasm-Nucleus        0.013003
Peroxisome               0.011120
Name: proportion, dtype: float64


In [33]:
# split the training data into true training set and validation set, leaving the test set untouched

from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["subcellular_location"]
)

In [34]:
print(len(train_df))
print(len(val_df))
print(len(test_df))

10035
1116
2769
